In [1]:
%pip install -U langchain-openai
%pip install -U langchain-community
%pip install -U langgraph

  Using cached tiktoken-0.13.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.7 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.18-py3-none-any.whl.metadata (2.4 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached uuid_utils-0.17.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.4 kB)
  Using cached anyio-4.14.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.16.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.2 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import re
from dotenv import load_dotenv

from langgraph.graph import StateGraph, END
from typing import TypedDict

In [3]:
# LLMs
from langchain_openai import ChatOpenAI

In [4]:
load_dotenv()

True

### Definindo o modelo llm 

In [5]:
api_key = os.getenv('OPENAI_API_KEY')
gpt_llm_model = os.getenv('GPT_LLM_MODEL')

In [6]:
model = ChatOpenAI(
  api_key=api_key,
  model=gpt_llm_model,
  temperature=0
)

In [7]:
response = model.invoke('Hello World')

In [8]:
response.content

'Hello! How can I assist you today?'

### Definindo o Agente 

In [ ]:
class Agents:
  def __init__(self, system=""):
    self.system = system
    self.messages = []
    
    if self.system:
      self.messages.append({"role": "system", "content": self.system})
      
  def __call__(self, message):
    self.messages.append({"role": "user", "content": message})
    result = self.execute()
    self.messages.append({"role": "assistant", "content": result})
    return result
  
  def execute(self):
    # Constrói o promp com todo o histórico
    prompt = ""

    for msg in self.messages:
      prompt += f"{msg['role']}: {msg['content']}\n"
    
    response = model.invoke(prompt)
     
    return response.content

In [10]:
if __name__ == "__main__":
  agent = Agents(system="Você é um assistente útil e objetivo.")
  print(agent("Qual é a capital da franca?"))

A capital da França é Paris.


In [11]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.

Ações disponíveis:
    - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
    - consultar_preço_produto: retorna o preço unitário de um produto (ex: "consultar_preço_produto: mouse gamer")

Exemplo:

Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.
""".strip()

In [12]:
class AgentState(TypedDict):
  pergunta: str
  historico: list[str]
  acao_pendente: str
  resposta_final: str

In [13]:
def consultar_estoque(item: str) -> str:
  # Simula a consulta de estoque de um item no inventário
  
  item = item.lower()
  
  estoque = {
    "monitor": 75,
    "teclado": 120,
    "mouse gamer": 80,
    "webcam": 40,
    "headset": 60,
    "impressora": 15
  }
  
  if item in estoque:
    return f"Temos {estoque[item]} {item}s em estoque."
  
  else:
    return f"Item '{item}' não encontrado no inventário."

In [14]:
def consultar_preco_produto(produto: str) -> str:
  
  produto = produto.lower()
  
  precos = {
    "monitor": 999.90,
    "teclado": 150.00,
    "mouse gamer": 99.50,
    "webcam": 120.00,
    "headset": 180.00,
    "impressora": 750.00
  }
  
  if produto in precos:
    return f"O preço de um(a) {produto} é R$ {precos[produto]:.2f}."
  else:
    return f"Produto não encontrado na lista de preços."

In [15]:
print(consultar_estoque("teclado"))

Temos 120 teclados em estoque.


In [16]:
print(consultar_preco_produto("impressora"))

O preço de um(a) impressora é R$ 750.00.


In [ ]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
  model = ChatOpenAI(
    api_key=api_key,
    model=gpt_llm_model,
    temperature=0
  )
  
  messages = [
    {
      "role": "system",
      "content": PROMPT_REACT
    }
  ]
  
  current_prompt = pergunta
  
  for i in range(max_iterations):
    messages.append({ "role": "system", "content": current_prompt })
    response = model.invoke(messages)
    response_text = response.content.strip()
    
    messages.append({ "role": "assistant", "content": response_text })
    
    print(f"--- Iteração {i + 1} ---")
    print(f"Modelo pensou/respondeu:\n{response_text}\n")
    
    if response_text.startswith("Resposta:"):
      return response_text.replace("Resposta:", "", 1).strip()
    
    
    
    